# Analytics Calculation Notebook

### Mount Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

### Load in Ground Truth Data

In [ ]:
# Load in ground truth data
all_test_set_path = '/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/all_test_set.csv'
all_test_set_df = pd.read_csv(all_test_set_path)

In [ ]:
all_test_set_df.head()

,art_style,painting,emotion,utterance,repetition,embeddings
0,Abstract_Expressionism,aaron-siskind_chicago-1951,sadness,dark and gloomy,6,"[0.022602325305342674, 0.012659378349781036, 0..."
1,Abstract_Expressionism,aaron-siskind_chicago-1951,fear,looks like some sort of creature from a scary ...,6,"[0.023607272654771805, 0.009993321262300014, -..."
2,Abstract_Expressionism,aaron-siskind_chicago-1951,fear,ghostly figure,6,"[-0.02539161965250969, -0.03483005613088608, 0..."
3,Abstract_Expressionism,aaron-siskind_chicago-1951,something else,This reminds me of more edgy modern art. Kinda...,6,"[0.030760714784264565, -0.009310646913945675, ..."
4,Abstract_Expressionism,aaron-siskind_chicago-1951,contentment,The light grey looks like the exhaust of two r...,6,"[0.013573752716183662, 0.027834365144371986, -..."


### Load in Results Data

In [ ]:
# Load in results
results_path = '/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Results_With_Embeddings/llama_results_0shot_embedded.csv'
results_df = pd.read_csv(results_path)

In [ ]:
results_df.head()

,test_painting,demo_paintings,demo_utterances,predicted_emotion,generated_description,embeddings
0,hans-hofmann_nulli-secundus-1964,"['mark-tobey_untitled-2', 'gene-davis_autumn-1...",['The orange colour curve look like orange pee...,amusement,the red color is very striking and the abstrac...,"[-0.0056884377263486385, -0.020650217309594154..."
1,gene-davis_color-needles-1984,"['jean-paul-riopelle_lithograph-1968', 'elaine...",['This feels like it started off great but som...,amusement,The colorful and dynamic arrangement of the ve...,"[0.002751436084508896, 0.013987968675792217, 0..."
2,arshile-gorky_bound-in-sleep,"['joan-snyder_still-2011', 'dan-flavin_juan-gr...",['The colors compliment each other very well a...,awe,The overall effect of the painting is one of c...,"[-0.03939918056130409, -0.004653977695852518, ..."
3,pino-pinelli_pittura-b-g-1991,"['esteban-vicente_comstock-1962', 'mark-tobey_...","['THIS LOOKS PRETTY BAD TO ME', 'An overly pop...",Awe,It is a beautiful piece of art.,"[-0.009846155531704426, -0.005903374403715134,..."
4,pyotr-konchalovsky_still-life-1911,"['esteban-vicente_harriet-1984', 'howard-mehri...",['The bright colors and almost disorganization...,something else,"The colors are fun and balanced well, but the ...","[0.009020834229886532, -0.005264392588287592, ..."


### Calculating Cosine Similarity, Append To Results Dataframe

In [ ]:
import numpy as np
import pandas as pd
import ast
from sklearn.metrics.pairwise import cosine_similarity

####doing this for human-human agreement

In [ ]:
# ---------------------------------------------------------
# 1. Helper: convert "embeddings" cell -> np.array
#    Handles: list, string "[...]", already-ndarray, NaN
# ---------------------------------------------------------
def to_embedding_array(x):
    # Already a numpy array
    if isinstance(x, np.ndarray):
        return x

    # Already a Python list of numbers
    if isinstance(x, list):
        return np.array(x, dtype=float)

    # String representation: "[0.1, 0.2, ...]"
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            return np.array(parsed, dtype=float)
        except Exception as e:
            raise ValueError(f"Could not parse embedding string: {x[:80]}...") from e

    # Missing value
    if pd.isna(x):
        return np.array([], dtype=float)

    # Anything else is unexpected
    raise TypeError(f"Unexpected embedding type: {type(x)} with value {x}")

In [ ]:
# ---------------------------------------------------------
# 2. Apply conversion to BOTH dataframes
# ---------------------------------------------------------
all_test_set_df["embedding_vec"] = all_test_set_df["embeddings"].apply(to_embedding_array)

# (Optional but nice) Drop any GT rows with empty embeddings
#all_test_set_df = all_test_set_df[
#    all_test_set_df["embedding_vec"].apply(lambda a: isinstance(a, np.ndarray) and a.size > 0)
#]

# ---------------------------------------------------------
# 3. Group ground-truth rows by painting ID
# ---------------------------------------------------------
gt_groups = {pid: group for pid, group in all_test_set_df.groupby("painting")}

# ---------------------------------------------------------
# 4. Function to compute:
#    - max cosine similarity
#    - index of best-matching GT row (global index)
#    - the GT description text (change column name as needed)
# ---------------------------------------------------------
def get_best_human_match(row):
    painting = row["painting"]
    v = row["embedding_vec"]
    current_index = row.name  # <-- index to exclude

    if not isinstance(v, np.ndarray) or v.size == 0:
        return pd.Series({
            "max_cosine_similarity": np.nan,
            "best_other_index": None,
            "best_other_description": None
        })

    query_emb = v.reshape(1, -1)

    # no GT group?
    if painting not in gt_groups:
        return pd.Series({
            "max_cosine_similarity": np.nan,
            "best_other_index": None,
            "best_other_description": None
        })

    # get all other GT rows for this painting
    gt_df = gt_groups[painting]

    # remove empty embeddings
    gt_df = gt_df[gt_df["embedding_vec"].apply(lambda a: isinstance(a, np.ndarray) and a.size > 0)]

    # exclude itself
    gt_df = gt_df[gt_df.index != current_index]

    if gt_df.empty:
        return pd.Series({
            "max_cosine_similarity": np.nan,
            "best_other_index": None,
            "best_other_description": None
        })

    # stack embeddings
    gt_embs = np.stack(gt_df["embedding_vec"].values)

    # cosine similarity
    sim_scores = cosine_similarity(query_emb, gt_embs)[0]

    best_idx_local = int(sim_scores.argmax())
    best_sim = float(sim_scores[best_idx_local])
    best_row = gt_df.iloc[best_idx_local]

    return pd.Series({
        "max_cosine_similarity": best_sim,
        "best_other_index": best_row.name,
        "best_other_description": best_row["utterance"]  # change as needed
    })

# ---------------------------------------------------------
# 5. Apply to every model-generated result
# ---------------------------------------------------------
all_test_set_df[[
    "human_max_cosine_similarity",
    "human_best_other_index",
    "human_best_other_description"
]] = all_test_set_df.apply(get_best_human_match, axis=1)

####doing this for ai-human agreement

In [ ]:
# ---------------------------------------------------------
# 1. Helper: convert "embeddings" cell -> np.array
#    Handles: list, string "[...]", already-ndarray, NaN
# ---------------------------------------------------------
def to_embedding_array(x):
    # Already a numpy array
    if isinstance(x, np.ndarray):
        return x

    # Already a Python list of numbers
    if isinstance(x, list):
        return np.array(x, dtype=float)

    # String representation: "[0.1, 0.2, ...]"
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            return np.array(parsed, dtype=float)
        except Exception as e:
            raise ValueError(f"Could not parse embedding string: {x[:80]}...") from e

    # Missing value
    if pd.isna(x):
        return np.array([], dtype=float)

    # Anything else is unexpected
    raise TypeError(f"Unexpected embedding type: {type(x)} with value {x}")

In [ ]:
# ---------------------------------------------------------
# 2. Apply conversion to BOTH dataframes
# ---------------------------------------------------------
results_df["embedding_vec"] = results_df["embeddings"].apply(to_embedding_array)
all_test_set_df["embedding_vec"] = all_test_set_df["embeddings"].apply(to_embedding_array)

# (Optional but nice) Drop any GT rows with empty embeddings
#all_test_set_df = all_test_set_df[
#    all_test_set_df["embedding_vec"].apply(lambda a: isinstance(a, np.ndarray) and a.size > 0)
#]

# ---------------------------------------------------------
# 3. Group ground-truth rows by painting ID
# ---------------------------------------------------------
gt_groups = {pid: group for pid, group in all_test_set_df.groupby("painting")}

# ---------------------------------------------------------
# 4. Function to compute:
#    - max cosine similarity
#    - index of best-matching GT row (global index)
#    - the GT description text (change column name as needed)
# ---------------------------------------------------------
def get_best_gt_match(row):
    painting = row["test_painting"]
    v = row["embedding_vec"]

    # If this embedding is empty or invalid, skip
    if not isinstance(v, np.ndarray) or v.size == 0:
        return pd.Series({
            "max_cosine_sim": np.nan,
            "best_gt_index": None,
            "best_gt_description": None
        })

    gen_emb = v.reshape(1, -1)

    # If no ground truth exists for this painting
    if painting not in gt_groups:
        return pd.Series({
            "max_cosine_sim": np.nan,
            "best_gt_index": None,
            "best_gt_description": None
        })

    # Ground truth dataframe for this painting
    gt_df = gt_groups[painting]

    # Safety: filter out any empty embeddings in GT
    gt_df = gt_df[gt_df["embedding_vec"].apply(lambda a: isinstance(a, np.ndarray) and a.size > 0)]
    if gt_df.empty:
        return pd.Series({
            "max_cosine_sim": np.nan,
            "best_gt_index": None,
            "best_gt_description": None
        })

    gt_embs = np.stack(gt_df["embedding_vec"].values)  # shape (k, d)

    # Compute cosine similarity scores
    sim_scores = cosine_similarity(gen_emb, gt_embs)[0]  # shape (k,)

    # Get the best (highest) score
    best_idx_local = int(sim_scores.argmax())      # index within this group
    best_sim = float(sim_scores[best_idx_local])   # best similarity score
    best_gt_row = gt_df.iloc[best_idx_local]       # full GT row
    best_global_index = best_gt_row.name           # index in all_test_set_df

    return pd.Series({
        "max_cosine_similarity": best_sim,
        "best_gt_index": best_global_index,
        # CHANGE THIS to your actual GT text column name:
        "best_gt_description": best_gt_row["utterance"]
    })

# ---------------------------------------------------------
# 5. Apply to every model-generated result
# ---------------------------------------------------------
results_df[["max_cosine_similarity", "best_gt_index", "best_gt_description"]] = results_df.apply(get_best_gt_match, axis=1)


In [ ]:
results_df.head()

,test_painting,demo_paintings,demo_utterances,predicted_emotion,generated_description,embeddings,embedding_vec,max_cosine_similarity,best_gt_index,best_gt_description
0,hans-hofmann_nulli-secundus-1964,[],[],Something-Else,I feel like this painting is very abstract and...,"[-0.0195406936109066, -0.05804502218961716, 0....","[-0.0195406936109066, -0.05804502218961716, 0....",0.698429,101,The painting makes me feel like I am walking d...
1,gene-davis_color-needles-1984,[],[],Awe,The use of thin lines in a horizontal pattern ...,"[0.016316348686814308, 0.014112216420471668, 0...","[0.016316348686814308, 0.014112216420471668, 0...",0.663300,423,The center of the painting seems to be bright....
2,arshile-gorky_bound-in-sleep,[],[],Awe,The use of bright colors and the way the face ...,"[-0.014129390008747578, -0.02588188275694847, ...","[-0.014129390008747578, -0.02588188275694847, ...",0.758641,541,"the calm demeanor of the face, the chalk like ..."
3,pino-pinelli_pittura-b-g-1991,[],[],amusement,it looks like a donut with the white cream and...,"[-0.0293964222073555, 0.035130731761455536, -0...","[-0.0293964222073555, 0.035130731761455536, -0...",0.626463,766,The fruity donuts glued onto the wall is funny...
4,pyotr-konchalovsky_still-life-1911,[],[],Contentment,The shapes are simple and make me feel calm,"[-0.0008190866210497916, 0.009310310706496239,...","[-0.0008190866210497916, 0.009310310706496239,...",0.665668,694,i dont feel anything when I look at this


In [ ]:
gt_groups

{'aaron-siskind_chicago-1951':                 art_style                    painting         emotion  \
 0  Abstract_Expressionism  aaron-siskind_chicago-1951         sadness   
 1  Abstract_Expressionism  aaron-siskind_chicago-1951            fear   
 2  Abstract_Expressionism  aaron-siskind_chicago-1951            fear   
 3  Abstract_Expressionism  aaron-siskind_chicago-1951  something else   
 4  Abstract_Expressionism  aaron-siskind_chicago-1951     contentment   
 5  Abstract_Expressionism  aaron-siskind_chicago-1951             awe   
 
                                            utterance  repetition  \
 0                                    dark and gloomy           6   
 1  looks like some sort of creature from a scary ...           6   
 2                                     ghostly figure           6   
 3  This reminds me of more edgy modern art. Kinda...           6   
 4  The light grey looks like the exhaust of two r...           6   
 5        The textures are very appe

### Calculating Accuracy of Emotion Prediction and Append to Results Dataframe

####human-human

In [ ]:
# ---------------------------------------------------------
# Build painting → set of ALL human emotions
# (same as your original code)
# ---------------------------------------------------------
from collections import defaultdict

emotions_map = defaultdict(list)
for index, row in all_test_set_df.iterrows():
    painting = row['painting']
    emotion = str(row['emotion']).lower()
    emotions_map[painting].append(emotion)

In [ ]:
def compute_human_human_accuracy(row):
    painting = row["painting"]
    current_emotion = str(row["emotion"]).lower()

    all_emotions = [emo for emo in emotions_map[painting]]
    if current_emotion in all_emotions: all_emotions.remove(current_emotion)
    other_emotions = all_emotions

    # If no other annotators exist → undefined agreement
    if len(other_emotions) == 0:
        return np.nan

    # Does this human agree with at least one other human?
    return 1 if current_emotion in other_emotions else 0

In [ ]:
all_test_set_df["human_human_accuracy"] = all_test_set_df.apply(
    compute_human_human_accuracy,
    axis=1
)

In [ ]:
human_results_df = all_test_set_df.copy(deep=True)
human_results_df.rename(columns={'utterance': 'generated_description', 'emotion': 'predicted_emotion', 'painting': 'test_painting', 'human_max_cosine_similarity': 'max_cosine_similarity', 'human_human_accuracy': 'accuracy'}, inplace=True)

In [ ]:
human_results_df.head()

,art_style,test_painting,predicted_emotion,generated_description,repetition,embeddings,embedding_vec,max_cosine_similarity,human_best_other_index,human_best_other_description,accuracy
0,Abstract_Expressionism,aaron-siskind_chicago-1951,sadness,dark and gloomy,6,"[0.022602325305342674, 0.012659378349781036, 0...","[0.022602325305342674, 0.012659378349781036, 0...",0.658607,3,This reminds me of more edgy modern art. Kinda...,0
1,Abstract_Expressionism,aaron-siskind_chicago-1951,fear,looks like some sort of creature from a scary ...,6,"[0.023607272654771805, 0.009993321262300014, -...","[0.023607272654771805, 0.009993321262300014, -...",0.735977,2,ghostly figure,1
2,Abstract_Expressionism,aaron-siskind_chicago-1951,fear,ghostly figure,6,"[-0.02539161965250969, -0.03483005613088608, 0...","[-0.02539161965250969, -0.03483005613088608, 0...",0.735977,1,looks like some sort of creature from a scary ...,1
3,Abstract_Expressionism,aaron-siskind_chicago-1951,something else,This reminds me of more edgy modern art. Kinda...,6,"[0.030760714784264565, -0.009310646913945675, ...","[0.030760714784264565, -0.009310646913945675, ...",0.658607,0,dark and gloomy,0
4,Abstract_Expressionism,aaron-siskind_chicago-1951,contentment,The light grey looks like the exhaust of two r...,6,"[0.013573752716183662, 0.027834365144371986, -...","[0.013573752716183662, 0.027834365144371986, -...",0.534989,1,looks like some sort of creature from a scary ...,0


In [ ]:
overall_human_human_accuracy = human_results_df["accuracy"].mean()
print(f"Overall Human–Human Agreement: {overall_human_human_accuracy*100:.2f}%")

Overall Human–Human Agreement: 71.90%


####human-ai

In [ ]:
from collections import defaultdict

# Read in all__test set
#all_tests_df = pd.read_csv('/content/drive/Shareddrives/LLMs_Art_Project/Data/Embeddings/Ground_Truth_Embeddings/all_test_set.csv')


# Creating dictionary mapping paintings to utterances
emotions_map = defaultdict(set)
for index, row in all_test_set_df.iterrows():
    painting = row['painting']
    emotion = row['emotion'].lower()
    emotions_map[painting].add(emotion)

print(emotions_map)
print(len(emotions_map))

defaultdict(<class 'set'>, {'aaron-siskind_chicago-1951': {'something else', 'fear', 'sadness', 'contentment', 'awe'}, 'aki-kuroda_untitled-1998': {'excitement', 'something else', 'awe', 'contentment'}, 'atsuko-tanaka_untitled-1963': {'excitement', 'something else', 'disgust', 'amusement'}, 'brice-marden_etchings-to-rexroth-1-1986': {'something else', 'excitement', 'disgust', 'sadness', 'contentment', 'awe', 'anger'}, 'clyfford-still_ph-118-1947': {'excitement', 'something else', 'awe', 'amusement'}, 'esteban-vicente_balada-1959': {'fear', 'amusement'}, 'friedel-dzubas_echo-1958': {'something else', 'fear', 'anger'}, 'gerhard-richter_mediation': {'excitement', 'contentment', 'anger', 'amusement'}, 'hans-hofmann_bird-cage-variation-ii-1958': {'something else', 'fear', 'excitement', 'disgust', 'contentment', 'awe', 'amusement', 'anger'}, 'hans-hofmann_gloriamundi-1963': {'excitement', 'something else', 'sadness', 'amusement'}, 'hans-hofmann_little-cherry-1965': {'sadness', 'something els

In [ ]:
# Read in file
#results_df = pd.read_csv("/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Qwen/()_results_0shot.csv")

# Iterate over rows
for index, row in results_df.iterrows():
    painting = row["test_painting"]

    # Get possible correct emotions
    emotions = emotions_map[painting]

    # Safely convert predicted_emotion to string and then to lowercase
    predicted_emotion = str(row["predicted_emotion"]).lower()

    # added just now
    if predicted_emotion =="something-else": predicted_emotion = "something else"

    # Calculate accuracy
    if predicted_emotion in emotions:
        results_df.at[index, "accuracy"] = 1
    else:
        results_df.at[index, "accuracy"] = 0

    # print(f"Prediction: {predicted_emotion}. |{results_df.iloc[index]["accuracy"]}|  Emotions: {emotions}")

# Calculate Overall Accuracy
overall_accuracy = results_df["accuracy"].mean()
print(f"Overall Accuracy: {overall_accuracy*100:.2f}%")

Overall Accuracy: 59.00%


In [ ]:
results_df.head()

,test_painting,demo_paintings,demo_utterances,predicted_emotion,generated_description,embeddings,embedding_vec,max_cosine_similarity,best_gt_index,best_gt_description,accuracy
0,hans-hofmann_nulli-secundus-1964,[],[],Something-Else,I feel like this painting is very abstract and...,"[-0.0195406936109066, -0.05804502218961716, 0....","[-0.0195406936109066, -0.05804502218961716, 0....",0.698429,101,The painting makes me feel like I am walking d...,0.0
1,gene-davis_color-needles-1984,[],[],Awe,The use of thin lines in a horizontal pattern ...,"[0.016316348686814308, 0.014112216420471668, 0...","[0.016316348686814308, 0.014112216420471668, 0...",0.663300,423,The center of the painting seems to be bright....,0.0
2,arshile-gorky_bound-in-sleep,[],[],Awe,The use of bright colors and the way the face ...,"[-0.014129390008747578, -0.02588188275694847, ...","[-0.014129390008747578, -0.02588188275694847, ...",0.758641,541,"the calm demeanor of the face, the chalk like ...",1.0
3,pino-pinelli_pittura-b-g-1991,[],[],amusement,it looks like a donut with the white cream and...,"[-0.0293964222073555, 0.035130731761455536, -0...","[-0.0293964222073555, 0.035130731761455536, -0...",0.626463,766,The fruity donuts glued onto the wall is funny...,1.0
4,pyotr-konchalovsky_still-life-1911,[],[],Contentment,The shapes are simple and make me feel calm,"[-0.0008190866210497916, 0.009310310706496239,...","[-0.0008190866210497916, 0.009310310706496239,...",0.665668,694,i dont feel anything when I look at this,1.0


### Output new results_df to file

In [ ]:
# Outut new results_df to file
output_path = "/content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Final_Results/human_results_embeddedandmetrics.csv"
human_results_df.to_csv(output_path, index=False)
print("Saved results to:", output_path)

Saved results to: /content/drive/Shareddrives/LLMs_Art_Project/Data/Results/Final_Results/human_results_embeddedandmetrics.csv


## extra

In [ ]:
paintings_map = defaultdict(set)
for index, row in all_test_set_df.iterrows():
    painting = row['painting']
    emotion = row['emotion'].lower()
    paintings_map[emotion].add(painting)

print(paintings_map)

paintings_map_results = defaultdict(set)
for index, row in results_df.iterrows():
    painting = row['test_painting']
    emotion = row['predicted_emotion'].lower()
    if emotion =="something-else": emotion = "something else"
    paintings_map_results[emotion].add(painting)

print(paintings_map_results)

for emotion in paintings_map:
  print(emotion)
  print(len(paintings_map[emotion]))
  print(len(paintings_map_results[emotion]))

defaultdict(<class 'set'>, {'sadness': {'barnett-newman_onement-v-1952', 'aaron-siskind_chicago-1951', 'andre-pierre-arnal_pliage-folded-painting-1975', 'anne-appleby_appleby-2011', 'brice-marden_etchings-to-rexroth-1-1986', 'pablo-picasso_a-dream-1932', 'jack-tworkov_indian-red-series-i-1979', 'richard-tuttle_sand-tree-2-1988', 'henri-laurens_le-grand-adieu-1941', 'robert-ryman_untitled-1959', 'jackson-pollock_black-white-number-20-1951', 'hans-hofmann_little-cherry-1965', 'fernand-leger_skating-rink-drawing-of-the-curtain-of-scene-1921', 'myron-stout_untitled-5-2-17-55-1955', 'gene-davis_cannonball-1969', 'fernand-leger_the-study-for-the-city-centre-1927', 'paul-jenkins_phenomena-lands-end', 'albert-gleizes_untitled-2', 'pyotr-konchalovsky_still-life-1911', 'joan-snyder_perpetuo-eternally-2004', 'charles-hinman_dance-1972', 'mira-schendel_untitled-from-cadernos-series-1971', 'gene-davis_untitled-pink-stripes-gray-box-1976', 'ian-davenport_puddle-painting-magenta-2008', 'mario-sironi_